## Historical PM 2.5 Data

Downloading the historical data from the past 5 years

In [1]:
import pandas as pd

PARAMETER_CODE = "88101"  # PM2.5 FRM/FEM
YEARS = range(2021, 2026)

frames = []

for year in YEARS:
    url = (
        "https://aqs.epa.gov/aqsweb/airdata/"
        f"daily_{PARAMETER_CODE}_{year}.zip"
    )

    print(f"Downloading {year}...")
    df_year = pd.read_csv(url, low_memory=False)
    frames.append(df_year)

aqs = pd.concat(frames, ignore_index=True)

print(aqs.shape)
print(aqs.columns.tolist())

(4000854, 29)
['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Sample Duration', 'Pollutant Standard', 'Date Local', 'Units of Measure', 'Event Type', 'Observation Count', 'Observation Percent', 'Arithmetic Mean', '1st Max Value', '1st Max Hour', 'AQI', 'Method Code', 'Method Name', 'Local Site Name', 'Address', 'State Name', 'County Name', 'City Name', 'CBSA Name', 'Date of Last Change']


## Filtering to local (Triangle) zip codes

In [2]:
triangle_counties = {
    "Wake": "183",
    "Durham": "063",
    "Orange": "135",
}

triangle = aqs[
    (aqs["State Code"].astype(str).str.zfill(2) == "37")
    & (
        aqs["County Code"]
        .astype(str)
        .str.zfill(3)
        .isin(triangle_counties.values())
    )
].copy()

triangle["date"] = pd.to_datetime(triangle["Date Local"])

triangle = triangle.sort_values(
    ["State Code", "County Code", "Site Num", "POC", "date"]
)

print(triangle[
    [
        "date",
        "County Name",
        "Site Num",
        "POC",
        "Arithmetic Mean",
        "AQI",
        "Latitude",
        "Longitude",
    ]
].head())

             date County Name  Site Num  POC  Arithmetic Mean   AQI  \
483957 2021-01-01      Durham        15    3         5.956522   NaN   
484321 2021-01-01      Durham        15    3         5.900000  33.0   
483958 2021-01-02      Durham        15    3         5.347826   NaN   
484322 2021-01-02      Durham        15    3         5.300000  29.0   
483959 2021-01-03      Durham        15    3         4.583333   NaN   

         Latitude  Longitude  
483957  36.032955 -78.904037  
484321  36.032955 -78.904037  
483958  36.032955 -78.904037  
484322  36.032955 -78.904037  
483959  36.032955 -78.904037  


Saving as parquet to save space

In [4]:
triangle.to_parquet(
    "triangle_pm25_daily_2021_2025.parquet",
    index=False
)

Checking overall data for abnormalities

In [5]:
triangle[
    [
        "Sample Duration",
        "Pollutant Standard",
        "Event Type",
        "Observation Count",
    ]
].value_counts(dropna=False).head(20)

Sample Duration  Pollutant Standard  Event Type  Observation Count
24-HR BLK AVG    PM25 24-hour 2012   NaN         1                    5418
1 HOUR           NaN                 NaN         24                   4880
24-HR BLK AVG    PM25 24-hour 2012   Included    1                    1014
1 HOUR           NaN                 Included    24                    922
24 HOUR          PM25 24-hour 2012   NaN         1                     562
1 HOUR           NaN                 NaN         22                    183
                                                 21                    170
                                                 23                    106
                                                 20                     57
                                     Included    22                     41
                                                 23                     27
                                     NaN         19                     17
                                     Included    21                     16
                                     NaN         11                     15
                                                 9                      13
                                                 13                     11
                                                 7                      11
                                                 8                      10
                                                 12                      9
                                                 14                      8
Name: count, dtype: int64

In [6]:
pm25_continuous = triangle[
    (triangle["Sample Duration"] == "1 HOUR")
    & (triangle["Observation Count"] >= 18)
].copy()

pm25_continuous["date"] = pd.to_datetime(
    pm25_continuous["Date Local"]
)

In [7]:
pm25_continuous["Event Type"].value_counts(dropna=False)

,count
Event Type,
NaN,5418
Included,1014


In [8]:
pm25_continuous["exceptional_event"] = (
    pm25_continuous["Event Type"].notna()
).astype(int)

In [9]:
monitor_cols = [
    "State Code",
    "County Code",
    "Site Num",
    "POC",
]

duplicate_counts = (
    pm25_continuous
    .groupby(monitor_cols + ["date"])
    .size()
    .value_counts()
)

print(duplicate_counts)

1    6432
Name: count, dtype: int64


In [10]:
site_daily = (
    pm25_continuous
    .groupby(
        [
            "date",
            "County Code",
            "County Name",
            "Site Num",
            "Latitude",
            "Longitude",
        ],
        as_index=False
    )
    .agg(
        pm25=("Arithmetic Mean", "mean"),
        monitors=("POC", "nunique"),
        minimum_hour_count=("Observation Count", "min"),
        exceptional_event=("exceptional_event", "max"),
    )
)

In [11]:
triangle_daily = (
    site_daily
    .groupby("date", as_index=False)
    .agg(
        pm25=("pm25", "mean"),
        active_sites=("Site Num", "count"),
        exceptional_event=("exceptional_event", "max"),
    )
    .sort_values("date")
)

In [12]:
print(triangle_daily.head())
print(triangle_daily.tail())
print(triangle_daily["active_sites"].value_counts().sort_index())
print(triangle_daily["pm25"].describe())

        date      pm25  active_sites  exceptional_event
0 2021-01-01  4.264674             3                  1
1 2021-01-02  5.442794             3                  1
2 2021-01-03  3.394444             3                  1
3 2021-01-04  3.588095             3                  1
4 2021-01-05  8.616071             3                  1
           date      pm25  active_sites  exceptional_event
1821 2025-12-27  9.050000             3                  0
1822 2025-12-28  5.294444             3                  0
1823 2025-12-29  5.350000             3                  0
1824 2025-12-30  3.536805             3                  0
1825 2025-12-31  4.789583             3                  0
active_sites
1      30
2     230
3    1566
Name: count, dtype: int64
count    1826.000000
mean        7.899227
std         4.013908
min         0.709722
25%         5.221354
50%         7.271527
75%         9.743800
max        44.364160
Name: pm25, dtype: float64
